In [0]:
# Data Quality Framework
# Will apply here SCD Type after the data quality

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col, row_number, current_timestamp
from pyspark.sql.window import Window

In [0]:
# First i need to clean the data after that i will check all the data and than will is placed this into the silver layer

In [0]:
@dp.materialized_view(
    name="e_commerce.silver.customer_silver",
    comment="Cleaned and transformed customer data with latest customer records"
)
def customer_silver():

    # Step 1: Read Bronze data as batch
    customer_bronze = spark.read.table(
        "e_commerce.bronze.customer_bronze"
    )

    # Step 2: Data type correction before window
    customer_bronze = (
        customer_bronze
        .withColumn("updated_at", col("updated_at").cast("timestamp"))
        .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast("timestamp"))
    )

    # Step 3: Latest record for each customer
    window_spec = Window.partitionBy("customer_id").orderBy(
        col("updated_at").desc(),
        col("ingestion_timestamp").desc()
    )

    customer_window_function = (
        customer_bronze
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

    # Step 4: Data type correction and final columns
    customer_silver_df = (
        customer_window_function
        .withColumn("customer_id", col("customer_id").cast("string"))
        .withColumn("first_name", col("first_name").cast("string"))
        .withColumn("last_name", col("last_name").cast("string"))
        .withColumn("email", col("email").cast("string"))
        .withColumn("phone", col("phone").cast("string"))
        .withColumn("city", col("city").cast("string"))
        .withColumn("gender", col("gender").cast("string"))
        .withColumn("registration_date", col("registration_date").cast("date"))
        .withColumn("customer_segment", col("customer_segment").cast("string"))
        .withColumnRenamed(
            "ingestion_timestamp",
            "bronze_ingestion_timestamp"
        )
        .withColumn(
            "silver_ingestion_timestamp",
            current_timestamp()
        )
        .withColumn(
            "file_modification_timestamp",
            col("file_modification_timestamp").cast("timestamp")
        )
        .select(
            "customer_id",
            "first_name",
            "last_name",
            "email",
            "phone",
            "city",
            "gender",
            "registration_date",
            "customer_segment",
            "updated_at",
            "bronze_ingestion_timestamp",
            "silver_ingestion_timestamp",
            "file_modification_timestamp"
        )
    )

    return customer_silver_df
